# Facebook Denoiser

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from utils.clear_memory import clear_memory

warnings.filterwarnings('ignore')

In [ ]:
input_dir_Pitt = Path('../ad_detection/data/raw/Pitt-origin')
output_dir_Pitt = Path('../ad_detection/data/denoised/Pitt-origin-Denoiser')

control_files_Pitt = list((input_dir_Pitt / 'Control').glob('*.wav')) + list((input_dir_Pitt / 'Control').glob('*.mp3'))
dementia_files_Pitt = list((input_dir_Pitt / 'Dementia').glob('*.wav')) + list((input_dir_Pitt / 'Dementia').glob('*.mp3'))

input_dir_Lu = Path('../ad_detection/data/raw/Lu')
output_dir_Lu = Path('../ad_detection/data/denoised/Lu-Denoiser')

control_files_Lu = list((input_dir_Lu / 'Control').glob('*.wav')) + list((input_dir_Lu / 'Control').glob('*.mp3'))
dementia_files_Lu = list((input_dir_Lu / 'Dementia').glob('*.wav')) + list((input_dir_Lu / 'Dementia').glob('*.mp3'))

## Load Model

In [ ]:
from denoiser import pretrained

model_name = 'dns64'
model = pretrained.dns64()

# Device detection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

model = model.to(device)
model.eval()

target_sr = model.sample_rate  # denoiser model's built-in sample rate (typically 16000)
print(f"Model: {model_name}, Sample Rate: {target_sr}, Device: {device}")

## Denoise Function

In [ ]:
def denoise_audio(audio_path, model, device, target_sr):
    """
    Apply Facebook Denoiser for speech denoising
    """
    audio, sr = sf.read(str(audio_path))

    # Multi-channel to mono
    if len(audio.shape) == 2:
        audio = np.mean(audio, axis=1)

    # Resample to target sample rate
    if sr != target_sr:
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)

    audio = audio.astype(np.float32)

    # Normalize to [-1, 1]
    audio = np.clip(audio, -1.0, 1.0)

    # Convert to torch tensor: [batch, channels, time]
    wav = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        denoised = model(wav)

    # Back to numpy: [time]
    denoised_audio = denoised.squeeze().cpu().numpy()
    denoised_audio = np.clip(denoised_audio, -1.0, 1.0)

    return denoised_audio, target_sr

In [ ]:
def batch_denoise(files, output_subdir, model, device, target_sr, group_name):
    """
    Batch denoising (clears memory before and after each file)

    Args:
        files: List of audio files to process
        output_subdir: Output subdirectory
        model: denoiser model instance
        device: Compute device
        target_sr: Target sample rate
        group_name: Group name (for progress display)
    """
    # Create output directory
    output_subdir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    skip_count = 0
    fail_count = 0

    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        # Unify output as .wav format
        output_file = output_subdir / (audio_file.stem + '.wav')

        # Skip already processed files
        if output_file.exists():
            skip_count += 1
            continue

        try:
            clear_memory()

            # Denoise
            denoised_audio, sr = denoise_audio(audio_file, model, device, target_sr)

            # Save (16-bit integer format)
            sf.write(str(output_file), denoised_audio, sr, subtype='PCM_16')
            success_count += 1

            # ⚡ Clear memory immediately after processing
            del denoised_audio  # Free large array
            clear_memory()

        except Exception as e:
            fail_count += 1
            print(f"\nFailed: {audio_file.name}: {e}")
            # ⚡ Clear memory after failure as well
            clear_memory()

    # Print statistics
    print(f"\n{group_name} processing complete:")
    print(f"Success: {success_count}")
    print(f"Skipped: {skip_count}")
    print(f"Failed: {fail_count}")
    print(f"Total: {len(files)}")

## Pitt Denoise

In [ ]:
clear_memory()

batch_denoise(
    dementia_files_Pitt,
    output_dir_Pitt / 'Dementia',
    model,
    device,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files_Pitt,
    output_dir_Pitt / 'Control',
    model,
    device,
    target_sr=target_sr,
    group_name='Control'
)

## Lu Denoise

In [ ]:
clear_memory()

batch_denoise(
    dementia_files_Lu,
    output_dir_Lu / 'Dementia',
    model,
    device,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files_Lu,
    output_dir_Lu / 'Control',
    model,
    device,
    target_sr=target_sr,
    group_name='Control'
)